In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats

In [2]:
ghg_capita_df = pd.read_csv('data/EDGAR_GHG_per_capita.csv')
gain_df = pd.read_csv('data/gain.csv')
gdp_df = pd.read_csv('data/gdp_input.csv')

## Explorative Datenanalyse vor dem Merge der Daten

Bevor die drei Datensätze (GHG, ND-GAIN, GDP) zusammenführen, prüfen:
1. **Keys / Duplikate**: Eindeutigkeit der Identifikatoren
2. **Year-Coverage**: Zeitliche Abdeckung
3. **Missingness**: Fehlende Werte pro Land
4. **Einheiten und Skalen**: Bedeutung der Werte
5. **ISO3 Konsistenz**: Ländercode-Kompatibilität

In [3]:
ghg_capita_df.head()

,EDGAR Country Code,Country,1970,1971,1972,1973,1974,1975,1976,1977,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,PLW,Palau,188.706221,185.902651,182.795379,179.857415,177.637090,176.468760,176.379952,177.250012,...,58.869074,55.834919,57.768972,58.583811,59.736352,59.532420,63.157857,63.594898,63.566744,66.687436
1,FLK,Falkland Islands,77.834252,79.346001,79.036712,76.192701,76.984364,79.977450,80.478612,80.655945,...,56.698000,57.055386,56.455920,55.269538,56.275853,56.477484,56.002590,56.517160,57.144712,57.631652
2,QAT,Qatar,215.028226,227.532745,246.047371,256.469773,204.549115,177.296852,167.728771,142.585163,...,56.267499,54.448181,52.842095,52.097737,51.896831,50.389466,50.716353,50.581728,53.158520,54.539105
3,KWT,Kuwait,108.001698,103.895842,101.405019,86.898658,68.466609,54.519918,52.612724,47.379234,...,36.542684,36.776518,35.465597,35.822832,35.023872,33.478254,34.944407,36.097423,37.441495,38.136567
4,BHR,Bahrain,87.766986,85.428410,85.864642,87.512960,89.234190,75.826134,74.512153,79.489759,...,43.011175,41.566640,39.579730,37.733920,38.486698,37.252710,36.414091,36.447698,35.897242,35.126596


In [4]:
gain_df.head()

,ISO3,Name,1995,1996,1997,1998,1999,2000,2001,2002,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,AFG,Afghanistan,34.783530,34.775074,34.988812,35.293407,35.177507,35.065559,35.198269,35.335123,...,31.544162,31.795961,31.903926,31.346547,31.165179,31.910511,31.727814,32.833517,32.633596,32.765017
1,ALB,Albania,41.396494,41.379214,41.333451,41.100159,41.026585,41.381430,41.395906,41.602467,...,46.435235,47.225916,47.113945,47.064079,47.340870,47.412095,47.738609,48.443320,49.348682,49.747451
2,DZA,Algeria,45.208524,45.310608,44.711577,44.217693,44.233838,44.198576,44.255934,44.295123,...,44.982398,45.123443,45.471441,46.624614,46.353467,46.487537,47.531633,47.246477,47.391899,47.689392
3,AND,Andorra,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AGO,Angola,34.167486,34.149985,34.130968,34.111716,34.072943,34.030188,34.129440,34.436386,...,31.935004,31.790461,32.897104,33.213149,34.102874,34.774563,35.292144,36.595834,36.894235,37.043357


In [5]:
gdp_df.head()

,ISO3,Name,1995,1996,1997,1998,1999,2000,2001,2002,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,AFG,Afghanistan,813.550256,813.550256,813.550256,813.550256,813.550256,813.550256,747.688045,926.507941,...,2224.490748,2284.075848,2213.181441,2335.795862,2432.276701,2583.485332,2561.981761,2144.166570,2122.995815,2211.280635
1,ALB,Albania,2665.541729,2979.809006,2717.129042,3021.035865,3471.650410,3861.295877,4300.829136,4661.386482,...,11259.239658,11662.036260,12078.859008,12771.034440,13696.788562,14792.257224,14511.983862,16127.753258,19446.236663,21263.195659
2,DZA,Algeria,7746.716839,8052.768567,8128.798914,8502.888360,8776.442095,9186.815903,9543.520353,10080.182500,...,14694.143827,13807.351584,13437.846023,13493.560749,13727.126636,13892.533345,12676.513056,14496.865470,15836.094437,16824.487903
3,AND,Andorra,24934.382774,26541.581446,29121.603706,30072.727273,31589.958243,33458.118163,36895.440245,38787.052254,...,48505.601735,50733.239001,53110.142627,53084.863964,55244.655409,57475.069284,52095.841400,59332.202910,68470.075948,71730.668682
4,AGO,Angola,2695.805985,3013.332813,3178.494802,3254.244475,3262.975529,3326.779914,3427.853949,3824.284804,...,7990.274457,7119.726443,6843.735897,6992.728834,7347.799936,7528.382418,6450.749946,7408.126591,7924.888806,8040.702450


### 1. Datenstruktur & Identifikatoren

In [6]:
# Shape und Spalten der drei Datensätze
print("GHG per Capita (EDGAR)")
print(f"Shape: {ghg_capita_df.shape}")
print(f"Spalten: {list(ghg_capita_df.columns[:5])}... (erste 5)")
print(f"\nIdentifikator-Spalten:")
for col in ['EDGAR Country Code', 'Country']:
    print(f"  - {col}: {ghg_capita_df[col].nunique()} unique values")

GHG per Capita (EDGAR)
Shape: (210, 57)
Spalten: ['EDGAR Country Code', 'Country', '1970', '1971', '1972']... (erste 5)

Identifikator-Spalten:
  - EDGAR Country Code: 210 unique values
  - Country: 210 unique values


In [7]:
print("ND-GAIN")
print(f"Shape: {gain_df.shape}")
print(f"Spalten: {list(gain_df.columns[:5])}... (erste 5)")
print(f"\nIdentifikator-Spalten:")
for col in ['ISO3', 'Name']:
    print(f"  - {col}: {gain_df[col].nunique()} unique values")

ND-GAIN
Shape: (192, 31)
Spalten: ['ISO3', 'Name', '1995', '1996', '1997']... (erste 5)

Identifikator-Spalten:
  - ISO3: 192 unique values
  - Name: 192 unique values


In [8]:
print("GDP")
print(f"Shape: {gdp_df.shape}")
print(f"Spalten: {list(gdp_df.columns[:5])}... (erste 5)")
print(f"\nIdentifikator-Spalten:")
for col in ['ISO3', 'Name']:
    print(f"  - {col}: {gdp_df[col].nunique()} unique values")

GDP
Shape: (192, 31)
Spalten: ['ISO3', 'Name', '1995', '1996', '1997']... (erste 5)

Identifikator-Spalten:
  - ISO3: 192 unique values
  - Name: 192 unique values


In [9]:
# Prüfen auf Duplikate in den Identifier-Spalten
# GHG
ghg_dups = ghg_capita_df['EDGAR Country Code'].duplicated().sum()
print(f"GHG - Duplikate in 'EDGAR Country Code': {ghg_dups}")
if ghg_dups > 0:
    print(f"  Duplikate: {ghg_capita_df[ghg_capita_df['EDGAR Country Code'].duplicated(keep=False)][['EDGAR Country Code', 'Country']].head()}")

# ND-GAIN
gain_dups = gain_df['ISO3'].duplicated().sum()
print(f"ND-GAIN - Duplikate in 'ISO3': {gain_dups}")
if gain_dups > 0:
    print(f"  Duplikate: {gain_df[gain_df['ISO3'].duplicated(keep=False)][['ISO3', 'Name']].head()}")

# GDP
gdp_dups = gdp_df['ISO3'].duplicated().sum()
print(f"GDP - Duplikate in 'ISO3': {gdp_dups}")
if gdp_dups > 0:
    print(f"  Duplikate: {gdp_df[gdp_df['ISO3'].duplicated(keep=False)][['ISO3', 'Name']].head()}")

GHG - Duplikate in 'EDGAR Country Code': 0
ND-GAIN - Duplikate in 'ISO3': 0
GDP - Duplikate in 'ISO3': 0


### 2. Zeitliche Abdeckung

In [10]:
# Jahr-Spalten identifizieren (Spalten die wie "1995" aussehen)
def get_year_columns(df):
    """Findet alle Spalten die Jahr-Daten enthalten (z.B. '1995', '2000')"""
    year_cols = [c for c in df.columns if str(c).isdigit()]
    return sorted([int(y) for y in year_cols])

ghg_years = get_year_columns(ghg_capita_df)
gain_years = get_year_columns(gain_df)
gdp_years = get_year_columns(gdp_df)

# Überlappung
overlap = set(ghg_years) & set(gain_years) & set(gdp_years)
print(f"Überlappende Jahre (alle 3 Datensätze): {min(overlap) if overlap else 'N/A'} - {max(overlap) if overlap else 'N/A'}")
print(f"Anzahl überlappender Jahre: {len(overlap)}")
print(f"Jahre: {sorted(overlap)}")

Überlappende Jahre (alle 3 Datensätze): 1995 - 2023
Anzahl überlappender Jahre: 29
Jahre: [1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]


### 3. ISO3-Code Konsistenz & Überlappung
Auch wenn es anders genannt wird, sind die EDGAR Country Codes vom ghg Datensatz auch ISO3-Codes.

In [11]:
# ISO3-Code Überlappung zwischen den Datensätzen

ghg_codes = set(ghg_capita_df['EDGAR Country Code'].dropna().unique())
gain_codes = set(gain_df['ISO3'].dropna().unique())
gdp_codes = set(gdp_df['ISO3'].dropna().unique())

print(f"\nAnzahl Länder pro Datensatz:")
print(f"  GHG (EDGAR):  {len(ghg_codes)} Länder")
print(f"  ND-GAIN:      {len(gain_codes)} Länder")
print(f"  GDP:          {len(gdp_codes)} Länder")

# Überlappung
all_three = ghg_codes & gain_codes & gdp_codes
ghg_gain = ghg_codes & gain_codes
ghg_gdp = ghg_codes & gdp_codes
gain_gdp = gain_codes & gdp_codes

print(f"\nÜberlappung:")
print(f"  In ALLEN 3 Datensätzen:     {len(all_three)} Länder")
print(f"  In GHG + ND-GAIN:           {len(ghg_gain)} Länder")
print(f"  In GHG + GDP:               {len(ghg_gdp)} Länder")
print(f"  In ND-GAIN + GDP:           {len(gain_gdp)} Länder")

# Nur in einem Datensatz
only_ghg = ghg_codes - gain_codes - gdp_codes
only_gain = gain_codes - ghg_codes - gdp_codes
only_gdp = gdp_codes - ghg_codes - gain_codes
gain_not_ghg = gain_codes - ghg_codes

print(f"Nur in ND-GAIN:  {len(only_gain)} Länder")
print(f"Nur in GDP:      {len(only_gdp)} Länder")
print(f"Nur in GHG:      {len(only_ghg)} Länder")
print(only_ghg)
print(f"In ND-GAIN/GDP, aber nicht GHG:  {len(gain_not_ghg)} Länder")
print(gain_not_ghg)



Anzahl Länder pro Datensatz:
  GHG (EDGAR):  210 Länder
  ND-GAIN:      192 Länder
  GDP:          192 Länder

Überlappung:
  In ALLEN 3 Datensätzen:     182 Länder
  In GHG + ND-GAIN:           182 Länder
  In GHG + GDP:               182 Länder
  In ND-GAIN + GDP:           192 Länder
Nur in ND-GAIN:  0 Länder
Nur in GDP:      0 Länder
Nur in GHG:      28 Länder
{'AIA', 'EU27', 'MTQ', 'TWN', 'SCG', 'NCL', 'ABW', 'CYM', 'FRO', 'TCA', 'GUF', 'PRI', 'ESH', 'BMU', 'HKG', 'VGB', 'REU', 'MAC', 'COK', 'GRL', 'GLP', 'GIB', 'FLK', 'SPM', 'SHN', 'ANT', 'PYF', 'GLOBAL TOTAL'}
In ND-GAIN/GDP, aber nicht GHG:  10 Länder
{'TUV', 'SMR', 'NRU', 'AND', 'MCO', 'LIE', 'MHL', 'SRB', 'MNE', 'FSM'}


Im ghg_capita sind gewisse Länder zusammengenommen. z.B. MNE und SRB sind in GHG als SCG zusammengenommen oder LIE gehört dort zu CHE. -> Allenfalls in ND-Gain und GDP Datensätzen diese Zeilen zusammenaddieren? (Achtung, so wird wahrscheinlich ND-Gain verzogen?)

In ghg_capita ausserdem noch EU27 und Global Total vor mergen entfernen.

### 4. Missingness (Fehlende Werte)

In [12]:
# GHG
print("\nGHG per Capita (EDGAR):")
print(f"{ghg_capita_df.isnull().sum().sum()} fehlende Werte gefunden")

# ND-GAIN
print("\nND-GAIN:")
print(f"{gain_df.isnull().sum().sum()} fehlende Werte gefunden")

# GDP
print("\nGDP:")
print(f"{gdp_df.isnull().sum().sum()} fehlende Werte gefunden")


GHG per Capita (EDGAR):
0 fehlende Werte gefunden

ND-GAIN:
145 fehlende Werte gefunden

GDP:
116 fehlende Werte gefunden


In [13]:
# ND-GAIN: Zeilen mit mindestens einem fehlenden Wert
gain_missing_rows = gain_df[gain_df.isnull().any(axis=1)]
print(f"ND-GAIN - Zeilen mit fehlenden Werten: {len(gain_missing_rows)} von {len(gain_df)}")
display(gain_missing_rows)

print("\n" + "="*80 + "\n")

# GDP: Zeilen mit mindestens einem fehlenden Wert
gdp_missing_rows = gdp_df[gdp_df.isnull().any(axis=1)]
print(f"GDP - Zeilen mit fehlenden Werten: {len(gdp_missing_rows)} von {len(gdp_df)}")
display(gdp_missing_rows)

ND-GAIN - Zeilen mit fehlenden Werten: 5 von 192


,ISO3,Name,1995,1996,1997,1998,1999,2000,2001,2002,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
3,AND,Andorra,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98,LIE,Liechtenstein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,MCO,Monaco,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
143,KNA,Saint Kitts and Nevis,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
147,SMR,San Marino,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN




GDP - Zeilen mit fehlenden Werten: 4 von 192


,ISO3,Name,1995,1996,1997,1998,1999,2000,2001,2002,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
43,CUB,Cuba,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
88,PRK,"Korea, Democratic People's Repub",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98,LIE,Liechtenstein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,MCO,Monaco,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 5. Bereinigung
1. **Saint Kitts und Nevis** wird verworfen, da dazu keine ND-GAIN Daten vorliegen
2. **Serbien** und **Montenegro** sind bei Treibhausgasemissionen zusammengefasst (SCG). Daher werden die beiden Länder in den anderen Datensätzen ebenfalls zusammengefasst (jeweils gemittelt).
3. Folgende Staaten haben keine ND-Gain Daten und sind gleichzeitig im Treibhausgasemissions-Datensatz mit einem grösseren Land zusammengefasst:
    - **Andorra (mit ESP)**
    - **Liechtenstein (mit CHE)**
    - **Monaco (mit FRA)**
    - **San Marino (mit ITA)**
    Diese Länder werden aus dem ND-GAIN Datensatz entfernt und im Treibhausgasemissions-Datensatz werden die Länder umbenannt, sodass sie nur nach den grösseren Ländern benannt sind (z.B. "Spanien und Andorra" wird zu "Spanien").
4. **EU27** und **Global Total** werden aus dem Treibhausgasemissions-Datensatz entfernt.
5. Bei **PRK** und **CUB** werden im GDP Datensatz NA's eingefügt.
6. Folgende Staaten werden aus ND-Gain Daten und GDP Daten gelöscht, da dazu keine GHG Daten vorhanden sind: **NRU, FSM, TUV, MHL**
7. Folgende Staaten werden entfernt, da sie nur in den GHG Daten enthalten sind: **VGB, HKG, NCL, PRI, PYF, MAC, SHN, CYM, GIB, SPM, COK, GLP, MTQ, BMU, GUF, FRO, TWN, ESH, TCA, ABW, FLK, ANT, GRL, AIA, REU**

HINWEIS: Die Bereinigung und der Merge wurden in preprocess_and_merge.py ausgelagert, sodass zukünftig das neu erstellte merged_df.csv einfach geladen werden kann.